# RoadNetra AI · Model B — Severe Accident Detection (Kaggle data)

End-to-end: **dataset → cleaning & split → preprocessing → training → best epoch → best confidence → accuracy & all metrics → export**.

**Data (Kaggle, public, downloaded automatically — no login):**

| Source | What we take |
|---|---|
| [amedeograndi/accidents-detection-dataset](https://www.kaggle.com/datasets/amedeograndi/accidents-detection-dataset) (CC BY 4.0) | `severe` class — rollovers, overturned vehicles, pile-ups, crushed cars |
| [marslanarshad/car-accidents-and-deformation-datasetannotated](https://www.kaggle.com/datasets/marslanarshad/car-accidents-and-deformation-datasetannotated) | `severe` + `totaled` (on fire, flipped, crushed); `no_accident` → negatives |
| [mehwishtahir722/accident-and-nonaccident-dataset-for-yolo](https://www.kaggle.com/datasets/mehwishtahir722/accident-and-nonaccident-dataset-for-yolo) | CCTV-angle (simulated) accidents; non-accident frames → negatives |
| [rahat52/traffic-density-singapore](https://www.kaggle.com/datasets/rahat52/traffic-density-singapore) | Real LTA traffic-camera frames (medium / high / jam) → negatives: stopped traffic is **not** a crash |

All severe classes → **class 0 `severe_accident`** (RoadNetra contract). Minor / moderate images are excluded, not relabelled.

**Result:** ~1,815 images (≈1,585 severe accident + ≈230 no-accident), 70 / 15 / 15 split.

**Why we rebuild the split:** every source ships flipped / rotated / greyscale / noised copies of the same photo (one source is
~3.5× inflated), and 35 % of one source's validation set were copies of training images. Re-using those splits would inflate every metric.

**Run:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.

In [ ]:
# 1 · Environment
!nvidia-smi -L
!pip install -q "ultralytics>=8.1.0" albumentations
import torch, ultralytics
print('torch', torch.__version__, '| ultralytics', ultralytics.__version__)
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > T4 GPU'

In [ ]:
# 2 · Configuration
EPOCHS_MAX  = 100     # upper bound; early stopping picks the real number
PATIENCE    = 20      # stop if val fitness hasn't improved for this many epochs
MODEL_ARCH  = 'yolov8s.pt'
IMGSZ, BATCH, SEED = 640, 16, 0
WEATHER_FRACTION = 0.25   # train-only rain / fog / blur / glare copies
PIPELINE_CONF = 0.55      # threshold used by RoadNetra's temporal filter

SAVE_TO_DRIVE = True
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT = '/content/drive/MyDrive/roadnetra_accident'
else:
    PROJECT = '/content/roadnetra_accident'
RUN = 'kaggle_yolov8s_v1'
DATA_DIR = '/content/accident_dataset'
# Optional: upload accident_dataset_v3.zip to Drive and point here to skip the ~15 min rebuild
PREBUILT_ZIP = '/content/drive/MyDrive/accident_dataset_v3.zip'

In [ ]:
%%writefile /content/build_accident_dataset.py
"""Builds the Model B (severe_accident) dataset from public Kaggle sources.

Pipeline:
  1. Download source zips from Kaggle (public, no login needed).
  2. Keep images whose boxes are severe accidents (rollover / crushed / fire / pile-up);
     images with only intact vehicles become negatives with empty labels.
  3. Remove corrupt images, clip boxes to the frame, drop slivers.
  4. Collapse flipped / rotated / greyscale copies of the same photo to one image.
  5. Group near-identical frames (video sequences) so a scene never spans two splits.
  6. Sample to the target size, split 70/15/15 by group, resize to <=640 px, write YOLO layout.

Usage:
    python training/build_accident_dataset.py --out data/accident_dataset
"""
import argparse
import json
import pickle
import random
import shutil
import urllib.request
import zipfile
from collections import Counter, defaultdict
from pathlib import Path

import cv2
import numpy as np
import yaml

KAGGLE_URL = "https://www.kaggle.com/api/v1/datasets/download/{}"

# pos: classes that count as severe_accident (remapped to 0)
# conflict: lesser-damage classes; an image containing them is skipped, since dropping
#           their boxes would teach the model that a damaged car is background
# negative: classes that mark intact vehicles; images with only these become empty-label negatives
SOURCES = {
    "amedeo": dict(
        kaggle="amedeograndi/accidents-detection-dataset",
        names={0: "moderate", 1: "severe"},
        pos={1}, conflict={0}, negative=set(),
        quota_pos=1100, quota_neg=0,
    ),
    "marsl": dict(
        kaggle="marslanarshad/car-accidents-and-deformation-datasetannotated",
        names={0: "no_accident", 1: "minor", 2: "moderate", 3: "severe", 4: "totaled_fire_flipped"},
        # its no_accident frames include pre/post-crash video frames, so they are not used as negatives
        pos={3, 4}, conflict={1, 2}, negative=set(),
        quota_pos=450, quota_neg=0,
    ),
    "mehwish": dict(
        kaggle="mehwishtahir722/accident-and-nonaccident-dataset-for-yolo",
        names={0: "accident", 1: "non_accident"},
        # simulated CCTV; the only clean negatives found, so pos is matched to stop
        # "synthetic look = no accident" becoming a shortcut
        pos={0}, conflict=set(), negative={1},
        quota_pos=200, quota_neg=210,
    ),
    "sg": dict(
        # real Singapore LTA traffic cameras, hand-labelled by density; heavy traffic / jams are the
        # key false-alarm case ("stopped cars are not a crash"). Image-level labels only -> all negatives.
        kaggle="rahat52/traffic-density-singapore",
        names={}, pos=set(), conflict=set(), negative=set(),
        negative_folders={"Medium", "High", "Traffic Jam"},
        camera_sim=0.85,   # fixed cameras: group each camera's shots so a camera never spans two splits
        quota_pos=0, quota_neg=150,
    ),
}

IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
SPLIT_RATIO = {"train": 0.70, "val": 0.15, "test": 0.15}
MAX_SIDE = 640
MIN_BOX_AREA = 0.0004      # boxes under 0.04 % of the frame are annotation noise
GROUP_BITS = 14            # images within this many of 256 hash bits (any flip/rotation, chained) share a split
COPY_BITS = 8              # within a group, an image this close to one already kept is a copy and is dropped
SCENE_SIM = 0.95          # thumbnail cosine similarity at/above which two images are the same scene


def download(slug, dst):
    if dst.exists() and zipfile.is_zipfile(dst):
        return dst
    print(f"Downloading {slug} ...")
    req = urllib.request.Request(KAGGLE_URL.format(slug), headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as r, open(dst, "wb") as f:
        shutil.copyfileobj(r, f, length=1 << 20)
    return dst


def fit_max_side(bgr):
    h, w = bgr.shape[:2]
    s = MAX_SIDE / max(h, w)
    return cv2.resize(bgr, (round(w * s), round(h * s)), interpolation=cv2.INTER_AREA) if s < 1 else bgr


def photo_bits(gray):
    """(8, 32) uint8: 256-bit average hash of a 16x16 thumbnail for each of the 8 flips/rotations."""
    g = cv2.resize(gray, (16, 16), interpolation=cv2.INTER_AREA).astype(np.float32)
    variants = []
    for k in range(4):
        r = np.rot90(g, k)
        for v in (r, r[:, ::-1]):
            variants.append(np.packbits(v > v.mean()))
    return np.stack(variants)


def photo_key(gray):
    """Exact flip/rotation/greyscale-invariant key (min over the 8 variants)."""
    return min(v.tobytes() for v in photo_bits(gray))


_POPCOUNT = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint16)


def hamming_min(bits_a, bits_b):
    """Smallest bit distance between image A's canonical hash and any flip/rotation of image B."""
    return int(_POPCOUNT[bits_a[0] ^ bits_b].sum(axis=1).min())


def hash_groups(bits, max_bits):
    """Union-find over images whose hashes differ by <= max_bits under any flip/rotation.
    Catches copies that were also brightened, noised or greyscaled before saving."""
    n = len(bits)
    uf = UnionFind(n)
    canon = bits[:, 0, :]                                   # (n, 32)
    for i in range(n - 1):
        d = _POPCOUNT[canon[i + 1:, None, :] ^ bits[i][None, :, :]].sum(axis=2).min(axis=1)
        for j in np.nonzero(d <= max_bits)[0]:
            uf.union(i, i + 1 + j)
    return [uf.find(i) for i in range(n)]


def scene_vector(gray):
    """Unit-norm 32x32 thumbnail; cosine similarity >= SCENE_SIM means the same scene / adjacent frame."""
    v = cv2.resize(gray, (32, 32), interpolation=cv2.INTER_AREA).astype(np.float32).ravel()
    v -= v.mean()
    return v / (np.linalg.norm(v) + 1e-6)


class UnionFind:
    def __init__(self, n):
        self.p = list(range(n))

    def find(self, a):
        while self.p[a] != a:
            self.p[a] = self.p[self.p[a]]
            a = self.p[a]
        return a

    def union(self, a, b):
        self.p[self.find(a)] = self.find(b)


def scene_groups(vectors, bits=None, camera=None, block=2048):
    """Union-find over pairs whose thumbnails are near-identical (blocked matrix product),
    plus pairs within GROUP_BITS hash bits when hashes are given. Chaining is intended:
    it only enlarges groups, and a group never spans two splits."""
    n = len(vectors)
    uf = UnionFind(n)
    if bits is not None:
        for i, g in enumerate(hash_groups(bits, GROUP_BITS)):
            uf.union(i, g)
    for s in range(0, n, block):
        sim = vectors[s:s + block] @ vectors.T
        for i, j in zip(*np.nonzero(sim >= SCENE_SIM)):
            if s + i < j:
                uf.union(s + i, j)
    # looser threshold among images of the same fixed-camera source (camera: list of (source, sim) or None)
    if camera is not None:
        for src in {c for c in camera if c}:
            idx = np.array([i for i, c in enumerate(camera) if c == src])
            sim = vectors[idx] @ vectors[idx].T
            for a, b in zip(*np.nonzero(sim >= src[1])):
                if a < b:
                    uf.union(int(idx[a]), int(idx[b]))
    return [uf.find(i) for i in range(n)]


def parse_labels(text, names):
    rows = []
    for line in text.splitlines():
        parts = line.split()
        if len(parts) < 5:
            continue
        cls = int(float(parts[0]))
        vals = [float(v) for v in parts[1:]]
        if len(vals) > 4:  # polygon -> bbox
            xs, ys = vals[0::2], vals[1::2]
            vals = [(min(xs) + max(xs)) / 2, (min(ys) + max(ys)) / 2, max(xs) - min(xs), max(ys) - min(ys)]
        rows.append((cls, *vals))
    return rows


def clean_box(x, y, w, h):
    x0, y0 = max(0.0, x - w / 2), max(0.0, y - h / 2)
    x1, y1 = min(1.0, x + w / 2), min(1.0, y + h / 2)
    w, h = x1 - x0, y1 - y0
    if w <= 0 or h <= 0 or w * h < MIN_BOX_AREA:
        return None
    return ((x0 + x1) / 2, (y0 + y1) / 2, w, h)


def scan_source(tag, cfg, zpath, report):
    """Returns candidate records (cached next to the zip, keyed on zip size/mtime and class config)."""
    st = Path(zpath).stat()
    key = (st.st_size, st.st_mtime, MIN_BOX_AREA, MAX_SIDE, "hash-after-resize",
           repr({k: cfg.get(k) for k in ("names", "pos", "conflict", "negative", "negative_folders")}))
    cache = Path(zpath).with_suffix(".scan.pkl")
    if cache.exists():
        cached = pickle.loads(cache.read_bytes())
        if cached["key"] == key:
            report["sources"][tag] = cached["stats"]
            print(f"  {tag}: {cached['stats']} (cached scan)")
            for r in cached["records"]:
                r["zip"] = str(Path(zpath).resolve())  # the cache may have been written from another folder
            return cached["records"]
    records, stats = _scan(tag, cfg, zpath)
    cache.write_bytes(pickle.dumps({"key": key, "stats": stats, "records": records}))
    report["sources"][tag] = stats
    print(f"  {tag}: {stats}")
    return records


def _scan(tag, cfg, zpath):
    z = zipfile.ZipFile(zpath)
    members = set(z.namelist())
    stats = Counter()
    records = []
    for img in sorted(m for m in members if m.lower().endswith(IMG_EXT)):
        if cfg.get("negative_folders") is not None:
            if img.split("/")[-2] not in cfg["negative_folders"]:
                stats["skipped (folder not used)"] += 1
                continue
            record = _record(tag, zpath, z, img, "neg", [], stats)
            if record:
                records.append(record)
            continue
        stem = img.rsplit(".", 1)[0]
        cands = [stem.replace("/images/", "/labels/") + ".txt", stem.replace("images/", "label/") + ".txt",
                 stem.replace("images/", "labels/") + ".txt", stem + ".txt"]
        lbl = next((c for c in cands if c in members), None)
        if lbl is None:
            stats["no label file"] += 1
            continue
        rows = parse_labels(z.read(lbl).decode("utf-8", "replace"), cfg["names"])
        classes = {r[0] for r in rows}
        if classes & cfg["pos"] and not classes & cfg["conflict"]:
            boxes = [b for b in (clean_box(*r[1:5]) for r in rows if r[0] in cfg["pos"]) if b]
            if not boxes:
                stats["positive but all boxes invalid"] += 1
                continue
            kind = "pos"
        elif classes and classes <= cfg["negative"]:
            boxes, kind = [], "neg"
        else:
            stats["skipped (lesser damage / unusable)"] += 1
            continue
        record = _record(tag, zpath, z, img, kind, boxes, stats)
        if record:
            records.append(record)
    return records, dict(stats)


def _record(tag, zpath, z, img, kind, boxes, stats):
    bgr = cv2.imdecode(np.frombuffer(z.read(img), np.uint8), cv2.IMREAD_COLOR)
    if bgr is None or min(bgr.shape[:2]) < 64:
        stats["corrupt / too small"] += 1
        return None
    bgr = fit_max_side(bgr)  # hash the image exactly as it will be saved
    b, g, r = cv2.split(bgr.astype(np.int16))
    colour = float(np.mean(np.abs(b - g)) + np.mean(np.abs(g - r)))
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    stats[kind] += 1
    return dict(tag=tag, zip=str(Path(zpath).resolve()), member=img, kind=kind, boxes=boxes,
                bits=photo_bits(gray), vec=scene_vector(gray), colour=colour)


def final_copy_sweep(out, counts):
    """Re-hash the saved JPEGs and drop any image within COPY_BITS of one already kept.
    JPEG re-encoding can nudge a borderline pair under the threshold; this makes the guarantee exact."""
    files = sorted((out / "images").glob("*/*.jpg"))
    bits = [photo_bits(cv2.imread(str(f), cv2.IMREAD_GRAYSCALE)) for f in files]
    kept, removed = [], 0
    for i, f in enumerate(files):
        if any(hamming_min(bits[i], bits[k]) <= COPY_BITS for k in kept):
            split, kind = f.parent.name, ("neg" if "_neg_" in f.name else "pos")
            lbl = out / "labels" / split / (f.stem + ".txt")
            counts[(split, "boxes")] -= len([l for l in lbl.read_text().splitlines() if l.strip()])
            counts[(split, kind)] -= 1
            f.unlink(); lbl.unlink()
            removed += 1
        else:
            kept.append(i)
    print(f"  post-save copy sweep removed {removed} images")
    return removed


def build(out, raw, seed=0):
    rng = random.Random(seed)
    raw.mkdir(parents=True, exist_ok=True)
    report = {"sources": {}, "dedupe": {}, "final": {}}

    records = []
    for tag, cfg in SOURCES.items():
        zpath = download(cfg["kaggle"], raw / (cfg["kaggle"].replace("/", "_") + ".zip"))
        records += scan_source(tag, cfg, zpath, report)

    print(f"De-duplicating {len(records)} candidates ...")
    # 1. exact copies (incl. flips / rotations / greyscale): keep the most colourful one
    by_key = defaultdict(list)
    for i, r in enumerate(records):
        by_key[min(v.tobytes() for v in r["bits"])].append(i)
    keep, conflicts = [], 0
    for idxs in by_key.values():
        if len({records[i]["kind"] for i in idxs}) > 1:  # same photo labelled accident and no-accident
            conflicts += 1
            continue
        keep.append(max(idxs, key=lambda i: records[i]["colour"]))
    records = [records[i] for i in keep]
    exact_unique = len(records)

    # 2. leakage groups: near copies and same-scene frames, chained; a group never spans two splits
    camera = [(r["tag"], SOURCES[r["tag"]]["camera_sim"]) if SOURCES[r["tag"]].get("camera_sim") else None for r in records]
    scene = scene_groups(np.stack([r["vec"] for r in records]), np.stack([r["bits"] for r in records]), camera)

    # 3. inside each group drop noisy / brightened copies of an image already kept (no chaining)
    members = defaultdict(list)
    for i, g in enumerate(scene):
        members[g].append(i)
    final, copies = [], 0
    for idxs in members.values():
        kept = []
        for i in sorted(idxs, key=lambda i: -records[i]["colour"]):
            near = [k for k in kept if hamming_min(records[i]["bits"], records[k]["bits"]) <= COPY_BITS]
            if not near:
                kept.append(i)
                continue
            copies += 1
            if any(records[k]["kind"] != records[i]["kind"] for k in near):
                conflicts += 1
                for k in near:
                    records[k]["conflict"] = True
        final += [k for k in kept if not records[k].get("conflict")]
    for i in final:
        records[i]["scene"] = scene[i]
    records = [records[i] for i in final]

    group_sizes = Counter(r["scene"] for r in records).values()
    report["dedupe"] = dict(candidates=sum(len(v) for v in by_key.values()), exact_unique=exact_unique,
                            near_copies_dropped=copies, conflicting_labels_dropped=conflicts, kept=len(records),
                            split_groups=len(group_sizes), largest_split_group=max(group_sizes))
    print(f"  {report['dedupe']}")

    # quota sampling, per source and kind
    chosen = []
    for tag, cfg in SOURCES.items():
        for kind, quota in (("pos", cfg["quota_pos"]), ("neg", cfg["quota_neg"])):
            pool = [r for r in records if r["tag"] == tag and r["kind"] == kind]
            rng.shuffle(pool)
            chosen += pool[:quota]
            report["final"][f"{tag}/{kind}"] = dict(available=len(pool), used=min(quota, len(pool)))

    # split by scene group so near-identical frames never cross splits
    scenes = defaultdict(list)
    for r in chosen:
        scenes[r["scene"]].append(r)
    groups = list(scenes.values())
    rng.shuffle(groups)
    groups.sort(key=len, reverse=True)  # place big groups first, then fill greedily
    target = {s: SPLIT_RATIO[s] * len(chosen) for s in SPLIT_RATIO}
    filled = Counter()
    for g in groups:
        split = max(SPLIT_RATIO, key=lambda s: (target[s] - filled[s]) / target[s])
        filled[split] += len(g)
        for r in g:
            r["split"] = split

    if out.exists():
        shutil.rmtree(out)
    zips = {}
    counts = Counter()
    for i, r in enumerate(chosen):
        z = zips.setdefault(r["zip"], zipfile.ZipFile(r["zip"]))
        bgr = fit_max_side(cv2.imdecode(np.frombuffer(z.read(r["member"]), np.uint8), cv2.IMREAD_COLOR))
        name = f"{r['tag']}_{r['kind']}_{i:05d}"
        (out / "images" / r["split"]).mkdir(parents=True, exist_ok=True)
        (out / "labels" / r["split"]).mkdir(parents=True, exist_ok=True)
        cv2.imwrite(str(out / "images" / r["split"] / f"{name}.jpg"), bgr, [cv2.IMWRITE_JPEG_QUALITY, 95])
        (out / "labels" / r["split"] / f"{name}.txt").write_text(
            "".join(f"0 {x:.6f} {y:.6f} {bw:.6f} {bh:.6f}\n" for x, y, bw, bh in r["boxes"]))
        counts[(r["split"], r["kind"])] += 1
        counts[(r["split"], "boxes")] += len(r["boxes"])

    swept = final_copy_sweep(out, counts)
    report["dedupe"]["copies_removed_after_saving"] = swept

    cfg = {"path": str(out.resolve()), "train": "images/train", "val": "images/val", "test": "images/test",
           "names": {0: "severe_accident"}}
    (out / "data.yaml").write_text(yaml.safe_dump(cfg, sort_keys=False))
    report["splits"] = {s: dict(accident_images=counts[(s, "pos")], negative_images=counts[(s, "neg")],
                                boxes=counts[(s, "boxes")]) for s in SPLIT_RATIO}
    report["total_images"] = sum(v["accident_images"] + v["negative_images"] for v in report["splits"].values())
    (out / "dataset_report.json").write_text(json.dumps(report, indent=2))

    print("\nSplit summary:")
    for s, v in report["splits"].items():
        print(f"  {s:5s}: {v['accident_images']:5d} accident + {v['negative_images']:4d} negative images, {v['boxes']:5d} boxes")
    print(f"  total: {report['total_images']} images  ->  {out / 'data.yaml'}")
    return out / "data.yaml"


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", default="data/accident_dataset", type=Path)
    ap.add_argument("--raw", default="data/raw_kaggle", type=Path)
    ap.add_argument("--seed", default=0, type=int)
    a = ap.parse_args()
    build(a.out, a.raw, a.seed)

## 3 · Dataset build — what happens
1. **Download** 3 Kaggle zips (~1.4 GB).
2. **Class filtering** — keep severe boxes → class 0; images with moderate/minor damage are skipped (dropping their boxes would teach the model that a wrecked car is background); intact-only images become **negatives** (empty label file).
3. **Cleaning** — corrupt / tiny images removed, boxes clipped to the frame, sliver boxes (< 0.04 % of frame) dropped, polygon labels converted to boxes.
4. **De-duplication** — exact copies (incl. flipped / rotated / greyscale) collapse to one; noised / brightened copies within 8 of 256 hash bits are dropped, and a final pass re-checks the saved JPEGs; photos labelled *accident* in one source and *no accident* in another are dropped.
5. **Leak-free split** — near copies, same-scene frames and each fixed traffic camera are grouped and a group never spans two splits; **70 % train / 15 % val / 15 % test**.
6. **Resize** — long side ≤ 640 px (YOLO letterboxes to 640×640 at train time; pixel normalisation 0–1 is done inside YOLO).

In [ ]:
# 3 · Build dataset
import sys, json, os, shutil, yaml
sys.path.insert(0, '/content')
from pathlib import Path
from build_accident_dataset import build
if os.path.exists(PREBUILT_ZIP):
    shutil.unpack_archive(PREBUILT_ZIP, '/content')
    DATA_YAML = Path(DATA_DIR) / 'data.yaml'
    cfg = yaml.safe_load(open(DATA_YAML)); cfg['path'] = DATA_DIR; yaml.safe_dump(cfg, open(DATA_YAML, 'w'), sort_keys=False)
    print('Using pre-built dataset from', PREBUILT_ZIP)
else:
    DATA_YAML = build(Path(DATA_DIR), Path('/content/raw_kaggle'), seed=SEED)
REPORT = json.load(open(f'{DATA_DIR}/dataset_report.json'))
print(json.dumps(REPORT, indent=1))

In [ ]:
# 4 · Exploratory data analysis
import glob, random, numpy as np, cv2, pandas as pd, matplotlib.pyplot as plt
def label_of(img):  # images/<split>/x.jpg -> labels/<split>/x.txt
    img = Path(img); return img.parents[2] / 'labels' / img.parent.name / (img.stem + '.txt')
rows = []
for split in ('train', 'val', 'test'):
    for lbl in glob.glob(f'{DATA_DIR}/labels/{split}/*.txt'):
        boxes = [list(map(float, l.split()[1:])) for l in open(lbl) if l.strip()]
        rows.append(dict(split=split, n_boxes=len(boxes), source=Path(lbl).stem.split('_')[0],
                         areas=[w * h for _, _, w, h in boxes]))
df = pd.DataFrame(rows)
df['type'] = np.where(df.n_boxes > 0, 'accident', 'negative')
display(df.pivot_table(index='split', columns='type', values='n_boxes', aggfunc='count', fill_value=0)
          .assign(total=lambda d: d.sum(axis=1)).loc[['train', 'val', 'test']])
display(df.pivot_table(index='source', columns='type', values='n_boxes', aggfunc='count', fill_value=0))

areas = np.concatenate([a for a in df.areas if a])
fig, ax = plt.subplots(1, 3, figsize=(18, 4))
df[df.n_boxes > 0].n_boxes.value_counts().sort_index().plot.bar(ax=ax[0], title='Boxes per accident image')
ax[1].hist(np.sqrt(areas), bins=40); ax[1].set_title('Box size (sqrt of area fraction)')
size_cls = pd.cut(areas, [0, 0.01, 0.09, 1], labels=['small <10%', 'medium', 'large >30%']).value_counts()
size_cls.plot.pie(ax=ax[2], autopct='%1.0f%%', title='Box size classes'); ax[2].set_ylabel('')
plt.show()

imgs = sorted(glob.glob(f'{DATA_DIR}/images/train/*'))
fig, axes = plt.subplots(3, 4, figsize=(20, 12))
for a, p in zip(axes.flat, random.Random(SEED).sample(imgs, 12)):
    im = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB); H, W = im.shape[:2]
    for l in open(label_of(p)):
        if l.strip():
            _, x, y, w, h = map(float, l.split())
            cv2.rectangle(im, (int((x-w/2)*W), int((y-h/2)*H)), (int((x+w/2)*W), int((y+h/2)*H)), (255, 0, 0), 3)
    a.imshow(im); a.set_title(Path(p).name, fontsize=8); a.axis('off')
plt.tight_layout(); plt.show()

## 5 · Preprocessing & augmentation
**Offline (train split only, so val/test stay real):** a `WEATHER_FRACTION` share of training images gets one extra copy with
rain, fog, motion blur, Gaussian blur, sun glare or low-light contrast — the 'Vision in the Wild' conditions (spec §3 B, D).

**Online (YOLO, every epoch, train only):** letterbox to 640×640, normalise to 0–1, HSV jitter (night / glare / smog),
rotation ±10° (pole sway), translate, scale ±50 %, perspective (high-mast angle), horizontal flip, mosaic (occlusion), mixup (clutter).

In [ ]:
# 5 · Offline weather copies (train only)
import albumentations as A, shutil
weather = A.OneOf([
    A.RandomRain(p=1), A.RandomFog(p=1), A.MotionBlur(blur_limit=(3, 9), p=1), A.GaussianBlur(p=1),
    A.RandomSunFlare(p=1), A.RandomBrightnessContrast(brightness_limit=(-0.5, 0.2), contrast_limit=(-0.4, 0.1), p=1),
], p=1)
train_imgs = [p for p in glob.glob(f'{DATA_DIR}/images/train/*.jpg') if '_pos_' in Path(p).name and not Path(p).name.startswith('wx_')]
made = 0
for p in random.Random(SEED).sample(train_imgs, int(len(train_imgs) * WEATHER_FRACTION)):
    out = weather(image=cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB))['image']
    q = Path(p).with_name('wx_' + Path(p).name)
    cv2.imwrite(str(q), cv2.cvtColor(out, cv2.COLOR_RGB2BGR))
    shutil.copy(label_of(p), label_of(q))
    made += 1
print(f'Weather copies added to train: {made}  → train now {len(glob.glob(f"{DATA_DIR}/images/train/*"))} images')

In [ ]:
# 6 · Train (early stopping finds the best epoch)
from ultralytics import YOLO
model = YOLO(MODEL_ARCH)
model.train(
    data=str(DATA_YAML), epochs=EPOCHS_MAX, patience=PATIENCE, imgsz=IMGSZ, batch=BATCH, device=0, seed=SEED,
    optimizer='auto', cos_lr=True, close_mosaic=10,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, degrees=10.0, translate=0.1, scale=0.5,
    perspective=0.0005, fliplr=0.5, mosaic=1.0, mixup=0.15,
    project=PROJECT, name=RUN, exist_ok=True,
)
# Colab disconnected? → YOLO(f'{PROJECT}/{RUN}/weights/last.pt').train(resume=True)

## 7 · Best epoch
YOLO scores each epoch with **fitness = 0.1·mAP50 + 0.9·mAP50-95** on the validation set and saves the best as `best.pt`.
Training stops when fitness hasn't improved for `PATIENCE` epochs. Watch the loss curves:
**train loss ↓ while val loss ↑ = overfitting** — the epoch where val loss bottoms out is the safe stopping point.

In [ ]:
# 7 · Epoch analysis
RUN_DIR = f'{PROJECT}/{RUN}'
res = pd.read_csv(f'{RUN_DIR}/results.csv'); res.columns = res.columns.str.strip()
res['fitness'] = 0.1 * res['metrics/mAP50(B)'] + 0.9 * res['metrics/mAP50-95(B)']
res['val/total_loss'] = res[['val/box_loss', 'val/cls_loss', 'val/dfl_loss']].sum(axis=1)
res['train/total_loss'] = res[['train/box_loss', 'train/cls_loss', 'train/dfl_loss']].sum(axis=1)
best = res.loc[res.fitness.idxmax()]
min_val = res.loc[res['val/total_loss'].idxmin()]
BEST_EPOCH = int(best.epoch) + (1 if res.epoch.min() == 0 else 0)
print(f'Epochs run          : {len(res)} (max {EPOCHS_MAX}, patience {PATIENCE})')
print(f'BEST EPOCH          : {BEST_EPOCH}  (mAP50 {best["metrics/mAP50(B)"]:.3f}, mAP50-95 {best["metrics/mAP50-95(B)"]:.3f})')
print(f'Lowest val loss at  : epoch {int(min_val.epoch) + (1 if res.epoch.min() == 0 else 0)}')

fig, ax = plt.subplots(1, 3, figsize=(20, 4.5))
ax[0].plot(res.epoch, res['train/total_loss'], label='train'); ax[0].plot(res.epoch, res['val/total_loss'], label='val')
ax[0].set_title('Total loss (box + cls + dfl)'); ax[0].legend()
for k in ('metrics/precision(B)', 'metrics/recall(B)'):
    ax[1].plot(res.epoch, res[k], label=k.split('/')[1])
ax[1].set_title('Precision / Recall per epoch'); ax[1].legend()
ax[2].plot(res.epoch, res['metrics/mAP50(B)'], label='mAP50'); ax[2].plot(res.epoch, res['metrics/mAP50-95(B)'], label='mAP50-95')
for a in ax: a.axvline(best.epoch, ls='--', c='grey'); a.set_xlabel('epoch')
ax[2].set_title(f'mAP per epoch (best = {BEST_EPOCH})'); ax[2].legend(); plt.show()

## 8 · Best confidence threshold
Every detection carries a **confidence score** (0–1). Low threshold → more crashes caught (recall ↑) but more false alarms
(precision ↓). The threshold that maximises **F1** balances both. RoadNetra additionally needs ≥3 consecutive frames above
`PIPELINE_CONF`, which suppresses one-frame false alarms, so a threshold a little above best-F1 is safe.

In [ ]:
# 8 · Confidence sweep on the validation set
best_model = YOLO(f'{RUN_DIR}/weights/best.pt')
val = best_model.val(data=str(DATA_YAML), split='val', imgsz=IMGSZ, plots=True, verbose=False)
px, p_c, r_c, f1_c = val.box.px, val.box.p_curve[0], val.box.r_curve[0], val.box.f1_curve[0]
i_best = int(np.argmax(f1_c)); BEST_CONF = float(px[i_best]); i_pipe = int(np.argmin(abs(px - PIPELINE_CONF)))
print(f'BEST CONFIDENCE (max F1): {BEST_CONF:.3f}  → P {p_c[i_best]:.3f}  R {r_c[i_best]:.3f}  F1 {f1_c[i_best]:.3f}')
print(f'At pipeline conf {PIPELINE_CONF}:     P {p_c[i_pipe]:.3f}  R {r_c[i_pipe]:.3f}  F1 {f1_c[i_pipe]:.3f}')
plt.figure(figsize=(8, 4.5))
plt.plot(px, p_c, label='Precision'); plt.plot(px, r_c, label='Recall'); plt.plot(px, f1_c, label='F1', lw=2.5)
plt.axvline(BEST_CONF, ls='--', c='k', label=f'best F1 @ {BEST_CONF:.2f}'); plt.axvline(PIPELINE_CONF, ls=':', c='r', label=f'pipeline {PIPELINE_CONF}')
plt.xlabel('confidence threshold'); plt.legend(); plt.title('Validation: metric vs confidence'); plt.show()

## 9 · Final test-set evaluation
The **test split was never used** for training, early stopping or threshold choice — these are the honest numbers to report.

* **Detection metrics** (box-level): Precision, Recall, F1, mAP50, mAP50-95.
* **Image-level metrics** (the operational question *"is there a severe accident in this frame?"*): an image is flagged if
  any box ≥ threshold. Gives Accuracy, Precision, Recall (sensitivity), Specificity, F1, False-alarm rate, Confusion matrix.

In [ ]:
# 9a · Detection metrics on the held-out test split
test = best_model.val(data=str(DATA_YAML), split='test', imgsz=IMGSZ, plots=True, verbose=False, name=f'{RUN}_test')
ti = int(np.argmin(abs(test.box.px - BEST_CONF)))
det = {
    'mAP50': float(test.box.map50), 'mAP50-95': float(test.box.map),
    'precision@best_conf': float(test.box.p_curve[0][ti]), 'recall@best_conf': float(test.box.r_curve[0][ti]),
    'F1@best_conf': float(test.box.f1_curve[0][ti]),
    'speed_ms': {k: round(v, 2) for k, v in test.speed.items()},
}
print(json.dumps(det, indent=1))

In [ ]:
# 9b · Image-level accident / no-accident accuracy on the test split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
test_imgs = sorted(glob.glob(f'{DATA_DIR}/images/test/*'))
y_true = [int(bool(open(label_of(p)).read().strip())) for p in test_imgs]
scores = [float(r.boxes.conf.max()) if len(r.boxes) else 0.0
          for r in best_model.predict(test_imgs, conf=0.001, imgsz=IMGSZ, verbose=False, stream=True)]

def image_metrics(th):
    y_pred = [int(s >= th) for s in scores]
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return dict(threshold=th, accuracy=(tp + tn) / len(y_true), precision=tp / max(1, tp + fp), recall=tp / max(1, tp + fn),
                specificity=tn / max(1, tn + fp), f1=2 * tp / max(1, 2 * tp + fp + fn), false_alarm_rate=fp / max(1, fp + tn),
                TP=int(tp), FP=int(fp), FN=int(fn), TN=int(tn)), y_pred

img_best, pred_best = image_metrics(BEST_CONF)
img_pipe, _ = image_metrics(PIPELINE_CONF)
display(pd.DataFrame([img_best, img_pipe]).set_index('threshold').round(3))
print(f'Test images: {len(y_true)} ({sum(y_true)} accident, {len(y_true) - sum(y_true)} negative)')
ConfusionMatrixDisplay(confusion_matrix(y_true, pred_best), display_labels=['no accident', 'severe accident']).plot(cmap='Blues')
plt.title(f'Test confusion matrix @ conf {BEST_CONF:.2f}'); plt.show()

In [ ]:
# 9c · Look at the mistakes (false alarms and misses)
wrong = [(p, t, s) for p, t, s in zip(test_imgs, y_true, scores) if int(s >= BEST_CONF) != t][:8]
if wrong:
    fig, axes = plt.subplots(2, 4, figsize=(20, 9))
    for a, (p, t, s) in zip(axes.flat, wrong):
        a.imshow(cv2.cvtColor(best_model.predict(p, conf=BEST_CONF, verbose=False)[0].plot(), cv2.COLOR_BGR2RGB))
        a.set_title(f'{"MISSED crash" if t else "FALSE ALARM"}  max conf {s:.2f}', fontsize=10); a.axis('off')
    plt.tight_layout(); plt.show()

In [ ]:
# 10 · Latency vs contract (<=25 ms GPU / <=75 ms CPU) and summary
import time
frame = np.random.randint(0, 255, (720, 1280, 3), dtype=np.uint8)
def bench(device, n):
    for _ in range(10): best_model.predict(frame, imgsz=IMGSZ, device=device, verbose=False)
    t = []
    for _ in range(n):
        s = time.perf_counter(); best_model.predict(frame, imgsz=IMGSZ, device=device, verbose=False); t.append((time.perf_counter() - s) * 1000)
    return float(np.median(t))
gpu_ms, cpu_ms = bench(0, 200), bench('cpu', 30)

SUMMARY = {
    'dataset': {'total_images': REPORT['total_images'], **{f'{k}_split': v for k, v in REPORT['splits'].items()}},
    'training': {'model': MODEL_ARCH, 'epochs_run': len(res), 'best_epoch': BEST_EPOCH, 'imgsz': IMGSZ, 'batch': BATCH},
    'best_confidence_threshold': round(BEST_CONF, 3),
    'test_detection': det,
    'test_image_level@best_conf': {k: (round(v, 4) if isinstance(v, float) else v) for k, v in img_best.items()},
    'test_image_level@pipeline_conf': {k: (round(v, 4) if isinstance(v, float) else v) for k, v in img_pipe.items()},
    'latency_ms': {'gpu_t4_median': round(gpu_ms, 1), 'colab_cpu_median': round(cpu_ms, 1)},
}
json.dump(SUMMARY, open(f'{RUN_DIR}/summary_metrics.json', 'w'), indent=2)
print(json.dumps(SUMMARY, indent=2))

In [ ]:
# 11 · Contract check + export
assert best_model.names == {0: 'severe_accident'}, best_model.names
shutil.copy(f'{RUN_DIR}/weights/best.pt', '/content/accident_best.pt')
shutil.make_archive('/content/accident_run_artifacts', 'zip', PROJECT)   # curves, confusion matrices, summary json
from google.colab import files
files.download('/content/accident_best.pt'); files.download('/content/accident_run_artifacts.zip')

## Glossary — the terms judges will ask about
| Term | Meaning |
|---|---|
| **Epoch** | One full pass over the training set. Too few → underfit; too many → overfit (memorises training images). |
| **Best epoch** | Epoch with the highest validation fitness (0.1·mAP50 + 0.9·mAP50-95); its weights are `best.pt`. |
| **Early stopping / patience** | Stop when validation fitness hasn't improved for *patience* epochs. |
| **Confidence score** | Model's certainty (0–1) that a box contains a severe accident. |
| **Confidence threshold** | Boxes below it are discarded. Best-F1 threshold balances misses vs false alarms. |
| **IoU** | Intersection-over-Union of predicted vs true box; ≥ 0.5 counts as a correct detection for mAP50. |
| **Precision** | Of all accident alerts raised, the share that were real. Low precision = false alarms. |
| **Recall (sensitivity)** | Of all real accidents, the share that were caught. Low recall = missed crashes. |
| **F1** | Harmonic mean of precision and recall — one number that punishes being bad at either. |
| **mAP50** | Area under the precision-recall curve at IoU 0.5 — main detection quality score. |
| **mAP50-95** | mAP averaged over IoU 0.50…0.95 — also rewards tight, accurate boxes. |
| **Accuracy** | (TP + TN) / all images — share of frames classified correctly as accident / no accident. Misleading if classes are imbalanced, so read it with precision/recall. |
| **Specificity** | Of all no-accident frames, the share correctly left alone. |
| **False-alarm rate** | FP / (FP + TN) = 1 − specificity. Traffic-jam frames wrongly flagged. |
| **TP / FP / FN / TN** | True positive (crash caught), false positive (false alarm), false negative (missed crash), true negative (quiet frame left alone). |
| **box / cls / dfl loss** | Training errors for box position, class score, and box-edge distribution — should all fall. |
| **Train / Val / Test** | Learn on train; tune epochs & threshold on val; report final numbers on untouched test. |
| **Latency** | Milliseconds per frame; 30 FPS needs ≤ 33 ms. |